In [1]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_ollama.llms import OllamaLLM
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain.chains.query_constructor.schema import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains import RetrievalQA
import json
import time

<h1>Define Models</h1>

<h3> RetrievalQA models </h3>

<h5>RAG Model 1</h5>

In [12]:
#all-minilm embedding model
embeddings = OllamaEmbeddings(model="all-minilm")

#beautiul soup parsed 10-K reports
db = Chroma(collection_name="k-10s", embedding_function=embeddings, persist_directory="../chroma_langchain_db-Nick/chroma_langchain_db")

#deepseek model
llm = OllamaLLM(model="deepseek-r1")

template = """Use the following context: {context} to answer the question: {question}
If you do not know the answer, just say "I don't know".  Don't make up any answers. Use 4 sentences maximum

Answer: Provide a concise answer to the question based on extracted information from the documents."""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)
qa_chain = RetrievalQA.from_chain_type(llm, retriever=db.as_retriever(search_kwargs={"k": 8}), return_source_documents=True, 
                                           chain_type_kwargs={"prompt": QA_CHAIN_PROMPT})

<h5> RAG Model 2 </h5>

In [3]:
#all-minilm embedding model
embeddings = OllamaEmbeddings(model="all-minilm")

#beautiul soup parsed 10-K reports
db = Chroma(collection_name="example_collection", embedding_function=embeddings, persist_directory="../chroma_semantic_chunking")

#deepseek model
llm = OllamaLLM(model="deepseek-r1")

template = """Use the following context: {context} to answer the question: {question}
If you do not know the answer, just say "I don't know".  Don't make up any answers. Use 4 sentences maximum

Answer: Provide a concise answer to the question based on extracted information from the documents."""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)
qa_chain_2 = RetrievalQA.from_chain_type(llm, retriever=db.as_retriever(search_kwargs={"k": 8}), return_source_documents=True, 
                                           chain_type_kwargs={"prompt": QA_CHAIN_PROMPT})

<h5> RAG Model 3 </h5>

In [4]:
#all-minilm embedding model
embeddings = OllamaEmbeddings(model="all-minilm")

#beautiul soup parsed 10-K reports
db = Chroma(collection_name="example_collection", embedding_function=embeddings, persist_directory="../chroma_langchain_db")

#deepseek model
llm = OllamaLLM(model="deepseek-r1")

template = """Use the following context: {context} to answer the question: {question}
If you do not know the answer, just say "I don't know".  Don't make up any answers. Use 4 sentences maximum

Answer: Provide a concise answer to the question based on extracted information from the documents."""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)
qa_chain_3 = RetrievalQA.from_chain_type(llm, retriever=db.as_retriever(search_kwargs={"k": 8}), return_source_documents=True, 
                                           chain_type_kwargs={"prompt": QA_CHAIN_PROMPT})

<h5> RAG Model 4 </h5>

In [5]:
#all-minilm embedding model
embeddings = OllamaEmbeddings(model="all-minilm")

#beautiul soup parsed 10-K reports
db = Chroma(collection_name="example_collection", embedding_function=embeddings, persist_directory="../chroma_semantic_chunking")

#deepseek model
llm = OllamaLLM(model="mistral-nemo")

template = """Use the following context: {context} to answer the question: {question}
If you do not know the answer, just say "I don't know".  Don't make up any answers. Use 4 sentences maximum

Answer: Provide a concise answer to the question based on extracted information from the documents."""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)
qa_chain_4 = RetrievalQA.from_chain_type(llm, retriever=db.as_retriever(search_kwargs={"k": 8}), return_source_documents=True, 
                                           chain_type_kwargs={"prompt": QA_CHAIN_PROMPT})

<h5> RAG Model 5 </h5>

In [6]:
#all-minilm embedding model
embeddings = OllamaEmbeddings(model="all-minilm")

#semantic chunking 10-K reports
db = Chroma(collection_name="example_collection", embedding_function=embeddings, persist_directory="../chroma_semantic_chunking")

#llama model model
llm = OllamaLLM(model="llama3.2")

template = """Use the following context: {context} to answer the question: {question}
If you do not know the answer, just say "I don't know".  Don't make up any answers. Use 4 sentences maximum

Answer: Provide a concise answer to the question based on extracted information from the documents."""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)
qa_chain_5 = RetrievalQA.from_chain_type(llm, retriever=db.as_retriever(search_kwargs={"k": 8}), return_source_documents=True, 
                                           chain_type_kwargs={"prompt": QA_CHAIN_PROMPT})

<h3> Self Query Retriever Models </h3>


In [2]:
class rag_10k:
    def __init__(self, metadata_field_info, document_content_description, vector_store,llm, prompt):
        self.metadata_field_info = metadata_field_info
        self.document_content_description = document_content_description
        self.vector_store = vector_store
        self.llm = llm 
        self.prompt = prompt
    def retrieve_context(self,question):
        #Define Retriever
        try:
            retriever = SelfQueryRetriever.from_llm(
                self.llm,
                self.vector_store,
                self.document_content_description,
                self.metadata_field_info,
            )
            retrieved_docs = retriever.invoke(question)
        except:
             retrieved_docs = self.vector_store.similarity_search(question)
        if(len(retrieved_docs) == 0):
             retrieved_docs = self.vector_store.similarity_search(question)
        return retrieved_docs
    def generate_response(self, question, context):
            docs_content = "\n\n".join(doc.page_content for doc in context)
            messages = self.prompt.invoke({"question": question, "context": docs_content})
            response = self.llm.invoke(messages)
            return response

In [ ]:
#Semantic Chunking on sections within document
embeddings_model = OllamaEmbeddings(model="all-minilm")
vector_store_semantic_sections = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings_model,
    persist_directory="../chroma_langchain_db"
)

In [ ]:
#Beautiful Soup Parsing 
embeddings_model = OllamaEmbeddings(model="all-minilm")
vector_store_manual = Chroma(
    collection_name="k-10s",
    embedding_function=embeddings_model,
    persist_directory="../chroma_langchain_db-Nick/chroma_langchain_db"
)

In [ ]:
#Semantic Chunking on Entire Document
embeddings_model = OllamaEmbeddings(model="all-minilm")
vector_store_semantic = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings_model,
    persist_directory="../chroma_semantic_chunking"
)

<h3> RAG Model 6 </h3>

In [ ]:
metadata_field_info = [
    AttributeInfo(
        name="ticker",
        description="Abbreviation of the company associated with the 10-K filing from which the text was extracted known as the ticker or stock symbol",
        type="string",
    )
]
document_content_description = "Collection of 10-K filings from various companies."


llm = OllamaLLM(model="llama3.2")

template = """Use the following context: {context} to answer the question: {question}
If you do not know the answer, just say "I don't know".  Don't make up any answers. Use 4 sentences maximum

Answer: Provide a concise answer to the question based on extracted information from the documents."""

prompt = ChatPromptTemplate.from_template(template)


In [7]:
rag_manual_model_6 = rag_10k(metadata_field_info=metadata_field_info,
              document_content_description=document_content_description,
              vector_store=vector_store_manual,
              llm = llm,prompt=prompt)


<h3>RAG Model 7</h3>

In [ ]:
metadata_field_info = [
    AttributeInfo(
        name="ticker",
        description="Abbreviation of the company associated with the 10-K filing from which the text was extracted known as the ticker or stock symbol",
        type="string",
    )
]
document_content_description = "Collection of 10-K filings from various companies."


llm = OllamaLLM(model="llama3.2")

template = """Use the following context: {context} to answer the question: {question}
If you do not know the answer, just say "I don't know".  Don't make up any answers. Use 4 sentences maximum

Answer: Provide a concise answer to the question based on extracted information from the documents."""

prompt = ChatPromptTemplate.from_template(template)


In [9]:
rag_semantic_model_7 = rag_10k(metadata_field_info=metadata_field_info,
              document_content_description=document_content_description,
              vector_store=vector_store_semantic,
              llm = llm,prompt=prompt)

<h5> Define Questions and Ground Trouth Responses/Tickers to Receive</h5>

In [ ]:

sample_queries = [
    #Targeted Answer
    "What company audited Amazon's 10-K report?",
    "When was the Iphone 12 released?",
    "Did Fox have more shareholders of Class A or Class B stock?",
    "Did Netflix settle any lawsuits in the document year?",
    "Who is Bank of America's CEO and how long have they been the CEO?",
    "How many daily active users did Facebook have on average in 2019?",
    "What sector of  Proctor and Gamble had the largest increase in net sales in 2020?",
    "How did Lululemon's net income in 2019 compare to 2018?",
    "Where are Microsoft's main product development facilities located?",
    "What are Uber's main segments?",

    #Strategic questions
    "What are the risks associated with Agilent?",
    "What are Bank of America's biggest risk factors?",
    "How does Google make money?",
    "What are NVDIA's key strategies?",
    "How is Salesforce planning on increasing sales?",
    "What are Costco's plans to mitigate the risk of rising labor cost and inflation?",
    "Where does most of General Motor's cash come from?",

    #Multi-Company Questions
    "What are companies doing to limit the risk of COVID?",
    "Compare the risks faced by TESLA to the risk GM is facing.",
    "Which company's stock is more stable, NIKE or Deckers Outdoor?"
]

expected_responses = [
    #Targeted Answer
    "Ernst & Young was the company responsible for auditing Amazon's 10-K report.",
    "The Iphone 12 was released in October 2020.",
    "Fox had more shareholders of Class A stock.  Fox shareholders held 343,678,951 shares of Class A Common Stock compared to 261,078,355 shares of Class B Common Stock were outstanding.",
    "Netflix is involved in litigation matters but they did not list them in the 10-K report as they were not considered material.",
    "Brian T. Moynihan is the Chief Executive Officer (CEO) and he has been the CEO since January 2010.",
    "Facebook's daily active users (DAUs) were 1.66 billion on average for December 2019",
    "Health Care saw the largest increase of Proctor and Gambles net sales in 2020, increasing by double digits.",
    "Net income increased $161.8 million, or 33%, to $645.6 million in fiscal 2019 from $483.8 million in fiscal 2018",
    "Microsoft's main procuct development facilities are located in Redmond, Washington",
    "Uber's five main operating segments are Rides, Eats, Freight, Other Bets, and Advanced Technologies Group (“ATG”)",

    #Strategic questions
    "The biggest risks to Agilent's business are the COVID-19 pandemic, markets where they sell their products decline or do not grow as anticipated, and failure to adjust to market conditions or introduce new products in a timely manner",
    "Bank of America biggest risk categories are market risk, liquidity risk, and credit risk. ",
    "The primary way Google makes money is through Brand and Performance advertising.",
    "NVDIA's key strategies include Advancing the GPU computing platform, Extending the technology and platform leadership in AI, Extending the technology and platform leadership in visual computing, Advancing the leading autonomous vehicle platform, and Leveraging their intellectual property",
    "Salesforce plans to increase sales by expanding relationship with existing customers, targeting vertical industries like financial services, healthcare and life sciences, government, manufacturing, consumer goods and philanthropy, and expanding into analytics, e-commerce, Internet of Things (“IoT”) integration."
    "What are Costco's plans to mitigate the risk of rising labor cost and inflation?",
    "General Motor's makes money through finance charge income, leasing income and proceeds from the sale of terminated leased vehicles, net distributions from credit facilities, securitizations, secured and unsecured borrowings and collections and recoveries on finance receivables",

    #Multi-Company Questions
    "Companies impose measures such as mandatory work from home policy for employees, restrictions on all non-essential travel and visitors, and following quarantine guidelines for employees who are necessary to be in the office.",
    "Both Tesla and GM face risks with the availability or price of the raw materials required to produce their vehicles that would impact their profatibility and political and economic risks associated with their international operations.  Teslas has experienced difficulty in their manufacturing process and keeping up with customer demand, which was not a key risk of GM.",
    "In Decker's 10-K report they state that their stock has been volatile leading to loss for the shareholders.  There was no such mention of this in the NIKE 10-K report, which would mean Nike stock is more stable. "

    
]

expected_tickers = [
    #Targeted Answer
    "AMZN",
    "AAPL",
    "FOX",
    "NFLX",
    "BAC",
    "META",
    "PG",
    "LULU",
    "MSFT",
    "UBER"

    #Strategic questions
    "A",
    "BAC",
    "GOOG",
    "NVDA",
    "CRM"
    "COST",
    "GM",

    #Multi-Company Questions
    "NA",
    ["GM","TSLA"],
    ["NKE","DECK"]
]

<h5> Generate Responses to Sample Questions to Store in a Dictionary </h5>

In [ ]:
def get_rag_comparison_dict(question_list, answer_list, expected_tickers, rag = None, qa_chain = None):
    results = {}
    result_list =[]
    for query_id, (question,gt_answer, ticker) in enumerate(zip(question_list, answer_list,expected_tickers)):
        query_dict = {}
        if rag:
            start_time = time.time()
            context = rag.retrieve_context(question)
            response = rag.generate_response(question, context)
            end_time = time.time()
            query_time = end_time - start_time

        elif qa_chain:
            start_time = time.time()
            response_dict = qa_chain.invoke(question)
            context = response_dict["source_documents"]
            try:
                response = response_dict['result'].split("</think>\n\n")[1].strip()
            except:
                response = response_dict['result']
            end_time = time.time()
            query_time = end_time - start_time
        retrieved_context = []
        for doc_id, doc in enumerate(context):
            doc_dict = {}
            doc_dict["doc_id"] = doc_id
            doc_dict["metadata"] = doc.metadata
            doc_dict["text"] = doc.page_content
            retrieved_context.append(doc_dict)

        query_dict["query_id"] = query_id
        query_dict["query"] = question
        query_dict["response"] = response
        query_dict["gt_answer"] = gt_answer
        query_dict["retrieved_context"] = retrieved_context
        query_dict["expected_ticker"] = ticker
        query_dict["query_time"] = query_time
        result_list.append(query_dict)
    results["results"] = result_list
    return results

RAG Model 1

In [ ]:
rag_comparison_model_1 = get_rag_comparison_dict(sample_queries,expected_responses,expected_tickers,qa_chain = qa_chain)
with open('./response_dicts/rag_responses_model_1.json', 'w') as json_file:
    json.dump(rag_comparison_model_1,json_file)

RAG Model 2

In [ ]:
rag_comparison_model_2 = get_rag_comparison_dict(sample_queries,expected_responses,expected_tickers,qa_chain = qa_chain_2)
with open('./response_dicts/rag_responses_model_2.json', 'w') as json_file:
    json.dump(rag_comparison_model_2,json_file)

RAG Model 3

In [ ]:
rag_comparison_model_3 = get_rag_comparison_dict(sample_queries,expected_responses,expected_tickers,qa_chain = qa_chain_3)
with open('./response_dicts/rag_responses_model_3.json', 'w') as json_file:
    json.dump(rag_comparison_model_3,json_file)

RAG Model 4

In [20]:
rag_comparison_model_4 = get_rag_comparison_dict(sample_queries,expected_responses,expected_tickers,qa_chain = qa_chain_4)
with open('./response_dicts/rag_responses_model_4.json', 'w') as json_file:
    json.dump(rag_comparison_model_4,json_file)

RAG Model 5

In [21]:
rag_comparison_model_5 = get_rag_comparison_dict(sample_queries,expected_responses,expected_tickers,qa_chain = qa_chain_5)
with open('./response_dicts/rag_responses_model_5.json', 'w') as json_file:
    json.dump(rag_comparison_model_5,json_file)

RAG Model 6

In [13]:
rag_comparison_model_6 = get_rag_comparison_dict(sample_queries,expected_responses,expected_tickers,rag = rag_manual_model_6)
with open('./response_dicts/rag_responses_model_6.json', 'w') as json_file:
    json.dump(rag_comparison_model_6,json_file)

RAG Model 7

In [14]:
rag_comparison_model_7 = get_rag_comparison_dict(sample_queries,expected_responses,expected_tickers,rag = rag_semantic_model_7)
with open('./response_dicts/rag_responses_model_7.json', 'w') as json_file:
    json.dump(rag_comparison_model_7,json_file)